[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/09_Kalman_Filter_and_Feedback_Control.ipynb)

# DiveLab

## Notebook 09 — Kalman Filter + Feedback Control

### From state estimation to realistic closed-loop control

**Guiding question:** What if the controller cannot see the true state and must control the diver using only an estimated state?

In previous notebooks we developed the pieces separately:

- buoyancy dynamics;
- instability and positive feedback;
- feedback control;
- delay;
- sensor noise and failure;
- observability;
- state observers;
- Kalman filtering.

Now we connect them into one closed loop.

## Learning objectives

By the end of this lab, you will be able to:

- distinguish full-state feedback from estimated-state feedback;
- use a Kalman filter inside a feedback loop;
- understand the architecture of an observer-based controller;
- compare ideal and realistic control;
- see how sensor noise propagates through estimation and control;
- understand the separation principle intuitively;
- recognize the structure that leads to LQG control.

# 1. The ideal controller

A state-feedback controller assumes that the complete state is known:

$$
u_k=-Kx_k.
$$

For our simplified diver:

$$
x_k=
\begin{bmatrix}
\delta z_k\\
\delta v_k
\end{bmatrix}.
$$

So the controller knows both depth deviation and vertical velocity exactly.

This is useful theoretically, but unrealistic.

A depth sensor does not directly reveal the complete state.

# 2. The realistic controller

Instead, the controller can use the estimated state:

$$
u_k=-K\hat x_k.
$$

The Kalman filter constructs $\hat x_k$ from:

- the physical model;
- the previous estimate;
- the applied control input;
- the noisy depth measurement.

So the information path becomes:

```text
                    noisy depth measurement
                              |
                              v
PLANT  ----------------->  SENSOR
  ^                           |
  |                           v
  |                     KALMAN FILTER
  |                           |
  |                     estimated state
  |                           |
  |                           v
  +------ ACTUATOR <------ CONTROLLER
```

This is **estimated-state feedback**.

## A crucial distinction

The physical plant evolves according to the **true state**:

$$
x_k.
$$

The controller acts according to the **estimated state**:

$$
\hat x_k.
$$

Therefore two dynamical processes evolve simultaneously:

1. the physical dynamics;
2. the estimation dynamics.

The controller succeeds only if both behave well.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# 3. Controlled linearized diver model

We use a two-state linear model:

$$
\dot x=Ax+Bu.
$$

The states are:

$$
x=
\begin{bmatrix}
\delta z\\
\delta v
\end{bmatrix}.
$$

The control input $u$ represents a simplified buoyancy-control action.

Positive $u$ produces an upward acceleration contribution.

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
gas_surface_volume_e = 0.005

def pressure_at_depth(z):
    return P0 + rho * g * z

dFb_dz = (
    -rho * g
    * gas_surface_volume_e
    * P0
    * rho * g
    / pressure_at_depth(z_e)**2
)

a_z = dFb_dz / mass

A = np.array([
    [0.0, -1.0],
    [a_z, 0.0]
])

B = np.array([
    [0.0],
    [1.0]
])

C = np.array([
    [1.0, 0.0]
])

print("A =")
print(A)
print("B =")
print(B)
print("C =")
print(C)
print("Open-loop eigenvalues =", np.linalg.eigvals(A))

The uncontrolled equilibrium has one stable and one unstable direction.

This is the saddle-point behavior developed earlier in DiveLab.

Feedback must move the closed-loop poles into the stable region.

# 4. Discrete-time model

The Kalman filter works naturally with:

$$
x_{k+1}=Fx_k+Gu_k+w_k.
$$

Using a small timestep:

$$
F\approx I+A\Delta t
$$

and:

$$
G\approx B\Delta t.
$$

In [ ]:
dt = 0.05

F = np.eye(2) + A * dt
G = B * dt
H = C.copy()

print("F =")
print(F)
print()
print("G =")
print(G)

# 5. Choose a stabilizing feedback controller

We use:

$$
u=-Kx
$$

with:

$$
K=
\begin{bmatrix}
k_z & k_v
\end{bmatrix}.
$$

Because positive velocity means upward motion while positive depth means downward displacement, the signs deserve attention.

We choose $K$ by discrete pole placement using a compact two-state calculation.

In [ ]:
# Desired continuous-time closed-loop poles
p1 = -0.7
p2 = -1.0

# For A - B K:
# characteristic polynomial:
# s^2 + kv*s + (kz - a_z)
kv = -(p1 + p2)
kz = p1 * p2 + a_z

Kc = np.array([[kz, kv]])

# Convert the same control law to the Euler-discretized model.
K = Kc.copy()

Acl_cont = A - B @ Kc
Fcl = F - G @ K

print("K =", K)
print("Continuous closed-loop eigenvalues =", np.linalg.eigvals(Acl_cont))
print("Discrete closed-loop eigenvalues =", np.linalg.eigvals(Fcl))

For a discrete system, stability requires the closed-loop eigenvalues to lie inside the unit circle:

$$
|\lambda_i|<1.
$$

# 6. First benchmark: perfect-state feedback

Before adding sensor noise and estimation, simulate an ideal controller that sees the true state:

$$
u_k=-Kx_k.
$$

In [ ]:
def simulate_perfect_feedback(
    x0,
    duration=20.0,
    dt=dt,
    u_limit=0.5
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    x = np.zeros((n, 2))
    u = np.zeros(n)

    x[0] = x0

    for k in range(n - 1):
        u[k] = float(np.clip(-(K @ x[k])[0], -u_limit, u_limit))
        x[k + 1] = F @ x[k] + G[:, 0] * u[k]

    u[-1] = u[-2]

    return t, x, u

x0 = np.array([0.5, 0.10])

t, x_perfect, u_perfect = simulate_perfect_feedback(x0)

In [ ]:
plt.plot(t, x_perfect[:, 0], label="Depth deviation")
plt.plot(t, x_perfect[:, 1], label="Vertical velocity")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("State")
plt.title("Perfect-state feedback")
plt.grid(True)
plt.legend()
plt.show()

This is our ideal benchmark.

The controller has impossible information: exact depth and exact velocity with no noise.

Realistic performance should be compared against this benchmark.

# 7. Add process and sensor uncertainty

The true plant now evolves as:

$$
x_{k+1}=Fx_k+Gu_k+w_k.
$$

The depth sensor reports:

$$
y_k=Hx_k+v_k.
$$

We model:

$$
w_k\sim\mathcal N(0,Q)
$$

and:

$$
v_k\sim\mathcal N(0,R).
$$

In [ ]:
Q = np.array([
    [1e-5, 0.0],
    [0.0, 5e-5]
])

sigma_sensor = 0.08

R = np.array([
    [sigma_sensor**2]
])

P0 = np.array([
    [0.25, 0.0],
    [0.0, 0.10]
])

# 8. The Kalman filter must know the control input

In Notebook 08 the prediction was:

$$
\hat x_k^- = F\hat x_{k-1}.
$$

Now the controller is actively changing the plant.

Therefore the estimator must include the known control action:

$$
\hat x_k^-
=
F\hat x_{k-1}+Gu_{k-1}.
$$

This is important.

If the estimator ignored its own actuator commands, it would interpret controlled motion as unexplained disturbance.

# 9. One complete closed-loop timestep

At each sample:

### Controller

$$
u_k=-K\hat x_k
$$

### Plant

$$
x_{k+1}=Fx_k+Gu_k+w_k
$$

### Sensor

$$
y_{k+1}=Hx_{k+1}+v_{k+1}
$$

### Kalman prediction

$$
\hat x_{k+1}^-
=
F\hat x_k+Gu_k
$$

$$
P_{k+1}^-
=
FP_kF^T+Q
$$

### Innovation

$$
r_{k+1}
=
y_{k+1}-H\hat x_{k+1}^-
$$

### Kalman correction

$$
\hat x_{k+1}
=
\hat x_{k+1}^-+K_{f,k+1}r_{k+1}.
$$

Here $K_f$ denotes the **Kalman filter gain**, to distinguish it from the controller gain $K$.

## Two different gains

We now have two matrices that play completely different roles.

### Controller gain $K$

Transforms state estimate into control action:

$$
u=-K\hat x.
$$

### Kalman gain $K_f$

Transforms measurement innovation into state-estimate correction:

$$
\hat x=\hat x^-+K_fr.
$$

One controls the **plant**.

The other corrects the **estimate**.

# 10. Implement the complete loop

In [ ]:
def simulate_kalman_feedback(
    x0,
    xhat0,
    duration=20.0,
    dt=dt,
    Q=Q,
    R=R,
    P0=P0,
    u_limit=0.5,
    seed=1
):
    rng_local = np.random.default_rng(seed)

    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    x = np.zeros((n, 2))
    xhat = np.zeros((n, 2))
    xpred = np.zeros((n, 2))

    y = np.zeros(n)
    u = np.zeros(n)
    innovation = np.zeros(n)

    P_hist = np.zeros((n, 2, 2))
    Kf_hist = np.zeros((n, 2))

    x[0] = x0
    xhat[0] = xhat0

    P = P0.copy()
    P_hist[0] = P

    y[0] = (H @ x[0])[0] + rng_local.normal(0.0, np.sqrt(R[0, 0]))

    I = np.eye(2)

    for k in range(n - 1):

        # ---------------------------------
        # 1. CONTROL USING ESTIMATED STATE
        # ---------------------------------
        u[k] = float(np.clip(-(K @ xhat[k])[0], -u_limit, u_limit))

        # ---------------------------------
        # 2. TRUE PLANT
        # ---------------------------------
        w = rng_local.multivariate_normal(np.zeros(2), Q)

        x[k + 1] = (
            F @ x[k]
            + G[:, 0] * u[k]
            + w
        )

        # ---------------------------------
        # 3. SENSOR
        # ---------------------------------
        y[k + 1] = (
            (H @ x[k + 1])[0]
            + rng_local.normal(0.0, np.sqrt(R[0, 0]))
        )

        # ---------------------------------
        # 4. KALMAN PREDICT
        # ---------------------------------
        x_minus = (
            F @ xhat[k]
            + G[:, 0] * u[k]
        )

        P_minus = F @ P @ F.T + Q

        # ---------------------------------
        # 5. INNOVATION
        # ---------------------------------
        r = y[k + 1] - (H @ x_minus)[0]
        S = (H @ P_minus @ H.T + R)[0, 0]

        # ---------------------------------
        # 6. KALMAN GAIN
        # ---------------------------------
        Kf = (P_minus @ H.T)[:, 0] / S

        # ---------------------------------
        # 7. CORRECT
        # ---------------------------------
        xhat[k + 1] = x_minus + Kf * r

        KH = np.outer(Kf, H[0])

        P = (
            (I - KH) @ P_minus @ (I - KH).T
            + np.outer(Kf, Kf) * R[0, 0]
        )

        xpred[k + 1] = x_minus
        innovation[k + 1] = r
        P_hist[k + 1] = P
        Kf_hist[k + 1] = Kf

    u[-1] = u[-2]

    return (
        t, x, xhat, xpred, y, u,
        innovation, P_hist, Kf_hist
    )

In [ ]:
xhat0 = np.array([0.0, 0.0])

(
    t,
    x_real,
    x_hat,
    x_pred,
    y_meas,
    u_est,
    innovation,
    P_hist,
    Kf_hist
) = simulate_kalman_feedback(
    x0=x0,
    xhat0=xhat0,
    seed=7
)

# 11. True depth, sensor measurement and estimated depth

In [ ]:
plt.plot(t, x_real[:, 0], label="True depth deviation")
plt.plot(t, y_meas, alpha=0.25, label="Noisy depth sensor")
plt.plot(t, x_hat[:, 0], label="Kalman estimate")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Estimated-state feedback: depth")
plt.grid(True)
plt.legend()
plt.show()

The controller never sees the true-depth curve.

It receives the Kalman estimate.

Yet the estimated state can still drive the system toward equilibrium.

# 12. Hidden vertical velocity

In [ ]:
plt.plot(t, x_real[:, 1], label="True vertical velocity")
plt.plot(t, x_hat[:, 1], label="Estimated vertical velocity")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Kalman reconstruction of unmeasured velocity")
plt.grid(True)
plt.legend()
plt.show()

Velocity is not directly measured.

It is reconstructed from:

- depth history;
- system dynamics;
- known control actions;
- uncertainty assumptions.

This is observability turned into a working estimator.

# 13. Compare ideal and realistic feedback

Now compare:

### Ideal

$$
u=-Kx
$$

### Realistic

$$
u=-K\hat x.
$$

In [ ]:
plt.plot(t, x_perfect[:, 0], label="Perfect-state feedback")
plt.plot(t, x_real[:, 0], label="Kalman estimated-state feedback")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Ideal vs estimated-state feedback")
plt.grid(True)
plt.legend()
plt.show()

The realistic loop will generally differ from the ideal loop because:

- measurements are noisy;
- the state estimate is imperfect;
- process disturbances are present;
- estimation takes time to converge.

But if estimation and control are well designed, the performance can remain close to the ideal benchmark.

# 14. Compare control effort

In [ ]:
plt.plot(t, u_perfect, label="Perfect-state control")
plt.plot(t, u_est, label="Estimated-state control")

plt.xlabel("Time [s]")
plt.ylabel("Control input")
plt.title("Control effort")
plt.grid(True)
plt.legend()
plt.show()

Notice that noisy measurements do not directly enter the controller.

They first pass through the estimator.

This prevents the controller from reacting blindly to every noisy sensor sample.

# 15. Estimation error during control

In [ ]:
e = x_real - x_hat

plt.plot(t, e[:, 0], label="Depth estimation error")
plt.plot(t, e[:, 1], label="Velocity estimation error")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Estimation error")
plt.title("Estimation error inside the closed loop")
plt.grid(True)
plt.legend()
plt.show()

The estimator has its own transient.

At the beginning, the estimate may be wrong.

As measurements arrive, the Kalman filter reconstructs the state while the controller simultaneously acts on the plant.

This is why estimator dynamics matter in closed-loop control.

# 16. The innovation

In [ ]:
plt.plot(t, innovation)

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Innovation [m]")
plt.title("Kalman innovation inside the feedback loop")
plt.grid(True)
plt.show()

The innovation remains the difference:

$$
r_k=y_k-H\hat x_k^-.
$$

It tells us whether the sensor agrees with what the model predicted after accounting for the known control input.

# 17. Separation principle

A remarkable result from linear systems theory is the **separation principle**.

Under the usual controllability and observability conditions, we can design:

1. the feedback controller;
2. the state estimator;

separately.

The controller gain $K$ determines the dynamics we want for the controlled plant.

The observer or Kalman filter determines how the state is reconstructed.

Then they can be combined.

## Why this is surprising

The controller acts on an estimate rather than the true state.

So one might expect controller design and estimator design to become inseparably mixed.

For linear systems, however, the combined closed-loop eigenstructure separates into:

- controller dynamics;
- estimator-error dynamics.

This is one of the most elegant results in control theory.

# 18. Deterministic separation: see the eigenvalues

To visualize the principle directly, use a Luenberger-style estimator:

$$
\hat x_{k+1}
=
F\hat x_k+Gu_k+L_d(y_k-H\hat x_k).
$$

The controller poles come from:

$$
F-GK.
$$

The estimator-error poles come from:

$$
F-L_dH.
$$

The combined system contains both sets.

In [ ]:
# Choose discrete observer poles
obs_poles = [0.82, 0.88]

# For the 2x2 discrete system, solve Ld numerically
# by matching trace and determinant of F - Ld H.

target_trace = sum(obs_poles)
target_det = np.prod(obs_poles)

# F - Ld H = [[F00-l1, F01],
#             [F10-l2, F11]]

l1 = F[0,0] + F[1,1] - target_trace

# determinant equation:
# (F00-l1)*F11 - F01*(F10-l2) = target_det
l2 = (
    target_det
    - (F[0,0] - l1) * F[1,1]
    + F[0,1] * F[1,0]
) / F[0,1]

Ld = np.array([[l1], [l2]])

controller_poles = np.linalg.eigvals(F - G @ K)
observer_poles_actual = np.linalg.eigvals(F - Ld @ H)

print("Controller poles:", controller_poles)
print("Observer poles:", observer_poles_actual)

The separation principle says that these two designs can be combined without redesigning them as one giant problem.

This provides the conceptual foundation for combining optimal control and optimal estimation.

# 19. From state feedback to LQR

So far we chose the controller poles directly.

Another approach is to choose the controller by minimizing a cost such as:

$$
J=
\sum_k
\left(
x_k^TQ_cx_k
+
u_k^TR_cu_k
\right).
$$

This balances:

- keeping the state near equilibrium;
- avoiding excessive control effort.

The resulting controller is the **Linear Quadratic Regulator**, or LQR.

## In simple words: what is LQR?

LQR asks:

> How strongly should I correct the diver when large errors are undesirable, but large control actions are also undesirable?

It turns that tradeoff into an optimization problem.

The result is still a state-feedback law:

$$
u=-K_{\mathrm{LQR}}x.
$$

# 20. From LQR + Kalman filter to LQG

Now combine:

### LQR

An optimal linear state-feedback controller.

### Kalman filter

An optimal linear state estimator under standard stochastic assumptions.

Together they form:

# Linear Quadratic Gaussian control — LQG

The controller becomes:

$$
u=-K_{\mathrm{LQR}}\hat x.
$$

## In simple words: what is LQG?

LQG says:

> Estimate the hidden state intelligently, then control the estimated state intelligently.

It combines two optimization problems:

- **Kalman filter:** best state estimate under the assumed stochastic model;
- **LQR:** best control action under the chosen quadratic cost.

The separation principle allows the two pieces to work together.

# 21. Important limitation

LQG is powerful, but our diver is not truly a linear Gaussian system.

Real buoyancy dynamics include:

- nonlinear pressure-volume relationships;
- nonlinear hydrodynamic drag;
- actuator saturation;
- delays;
- sensor faults;
- human actions;
- uncertain parameters.

So LQG is not the final answer.

It is a powerful theoretical framework that teaches us how estimation and control fit together.

# 22. Experiment: increase sensor noise

What happens when the depth sensor becomes much noisier?

In [ ]:
R_high = np.array([[0.30**2]])

(
    t_h,
    x_h,
    xhat_h,
    _,
    y_h,
    u_h,
    _,
    _,
    _
) = simulate_kalman_feedback(
    x0=x0,
    xhat0=xhat0,
    R=R_high,
    seed=7
)

plt.plot(t, x_real[:, 0], label="Baseline sensor")
plt.plot(t_h, x_h[:, 0], label="Noisier sensor")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Closed-loop robustness to sensor noise")
plt.grid(True)
plt.legend()
plt.show()

The Kalman filter can reduce the impact of sensor noise, but performance depends on the assumed $Q$ and $R$.

Estimation quality and control quality are connected through $\hat x$.

# 23. Experiment: wrong initial estimate

In [ ]:
xhat_bad = np.array([-0.8, -0.30])

(
    t_b,
    x_b,
    xhat_b,
    _,
    _,
    u_b,
    _,
    _,
    _
) = simulate_kalman_feedback(
    x0=x0,
    xhat0=xhat_bad,
    seed=7
)

plt.plot(t_b, x_b[:, 0], label="True depth")
plt.plot(t_b, xhat_b[:, 0], label="Estimated depth")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Closed loop with a poor initial estimate")
plt.grid(True)
plt.legend()
plt.show()

At first the controller may act incorrectly because the state estimate is wrong.

As the estimator converges, the controller receives better information.

This illustrates a general engineering principle:

> control performance depends on estimation performance.

# 24. Experiment: estimator trusts the model too much

Use a very small process-noise covariance $Q$.

The estimator becomes reluctant to believe that the plant can deviate from the model.

In [ ]:
Q_tiny = 1e-3 * Q

(
    t_q,
    x_q,
    xhat_q,
    _,
    _,
    u_q,
    _,
    _,
    _
) = simulate_kalman_feedback(
    x0=x0,
    xhat0=xhat0,
    Q=Q_tiny,
    seed=7
)

plt.plot(t, x_real[:, 0], label="Baseline Q")
plt.plot(t_q, x_q[:, 0], label="Very small Q")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Effect of estimator confidence in the model")
plt.grid(True)
plt.legend()
plt.show()

If the real plant differs from the assumed model, excessive confidence in the model can degrade estimation and therefore control.

This is the beginning of the topic of **robustness**.

# 25. The complete DiveLab control architecture

We can now describe the full conceptual system:

```text
                       ENVIRONMENT
                           |
                           v
                    +--------------+
          u ------> |    DIVER     | ------> true state x
                    +--------------+
                           |
                           v
                    DEPTH SENSOR
                           |
                     measurement y
                           |
                           v
                    KALMAN FILTER
                           |
                     estimate x_hat
                           |
                           v
                      CONTROLLER
                           |
                           v
                     control action u
```

External disturbances affect the physical diver.

Sensor noise affects the measurement.

Model uncertainty affects prediction.

The estimator reconstructs the state.

The controller acts on the estimate.

# 26. Connect the notebooks

We have now built a chain:

### Notebook 01
Pressure with depth.

### Notebook 02
Gas compression and buoyancy.

### Notebook 03
Positive feedback, instability and saddle point.

### Notebook 04
Feedback control.

### Notebook 05
Delay and stability.

### Notebook 06
Sensor noise and failure.

### Notebook 07
Observability and state estimation.

### Notebook 08
Kalman filtering.

### Notebook 09
Estimated-state feedback and LQG architecture.

The individual ideas now form a control system.

# Exercises

### 1. Increase sensor noise

Change:

```python
R
```

and observe:

- estimation error;
- depth response;
- control effort.

### 2. Change controller poles

Choose faster or slower desired poles.

How does aggressive control interact with imperfect state estimation?

### 3. Change the initial covariance

Try a very small and a very large:

```python
P0
```

How does initial confidence affect the first seconds of control?

### 4. Add a sensor bias

After 10 seconds, add:

```python
y += 0.3
```

to the depth sensor.

Does the Kalman filter protect the controller from a persistent bias?

### 5. Add control saturation

Reduce:

```python
u_limit
```

What happens when the controller asks for more correction than the actuator can provide?

# Challenge — combine delay, estimation and control

Notebook 05 studied delay.

Now introduce a delay between:

- measurement and Kalman correction;
- state estimate and controller action;
- controller command and actuator response.

Investigate whether a controller that works well without delay can become oscillatory when delay is added.

This reconnects the entire DiveLab story.

In [ ]:
# Your code here

# Summary

In this notebook we closed the loop using an estimated state.

The realistic control law is:

$$
u=-K\hat x.
$$

The Kalman filter estimates:

$$
\hat x
$$

from noisy measurements and a dynamical model.

We learned that:

- perfect-state feedback is an ideal benchmark;
- realistic controllers act on estimates;
- the estimator must know the applied control input;
- controller gain and Kalman gain have different roles;
- estimation and control evolve simultaneously;
- the separation principle lets us design estimator and controller separately for suitable linear systems;
- LQR optimizes state regulation versus control effort;
- Kalman filtering optimizes state estimation under standard stochastic assumptions;
- LQR + Kalman filter leads to LQG control.

### Core insight

> **A feedback controller does not need perfect knowledge of the physical state. It needs a sufficiently good estimate of that state.**

And this gives us the complete loop:

$$
\boxed{
\text{Plant}
\rightarrow
\text{Sensor}
\rightarrow
\text{Estimator}
\rightarrow
\text{Controller}
\rightarrow
\text{Plant}
}
$$

### Next

Notebook 10 can return to the **nonlinear diver model** and ask:

> How well do these linear control and estimation ideas survive when we restore Boyle's law, nonlinear drag, actuator saturation and realistic disturbances?

That would begin the transition from textbook linear control to a more realistic DiveLab simulator.